In [2]:
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='../../.env')

os.environ['OPENAI_API_KEY']= os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_ENDPOINT']= os.getenv('LANGSMITH_ENDPOINT')
os.environ['LANGSMITH_PROJECT']= os.getenv('LANGSMITH_PROJECT')
os.environ['LANGSMITH_TRACING']= os.getenv('LANGSMITH_TRACING')
os.environ['LANGSMITH_API_KEY']= os.getenv('LANGSMITH_API_KEY')


#문서 읽어오기, 임베딩, 벡터db
- 경로 아래에 있는 모든 txt파일.
- input_chunk_size : 청크 개당 길이
- input_chunk_overlao : 청크 오버랩
- filetype : 읽어오는 파일 타입
- len(docs) : 청크 개수
- model_name : 임베딩 모델 이름

In [3]:
import os
from glob import glob
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import DirectoryLoader

# 한국어 형태소 분할
from konlpy.tag import Okt

okt = Okt()

def len_okt(text):
    tokens = [token for token in okt.morphs(text)]

    return len(tokens)

def okt_tokenize(text):
    return [token for token in okt.morphs(text)]

# 텍스트 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

input_chunk_size = 1000
input_chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " ", ""],
    chunk_size = input_chunk_size,
    chunk_overlap = input_chunk_overlap,
    length_function = len_okt
)

In [4]:
file_path = "../../Data_Files"
filetype = 'txt'
print(f"디렉토리 경로'{file_path}' 아래 '{filetype}'형식의 파일들을 로드합니다.\n")

loader = DirectoryLoader(file_path ,glob="*."+filetype, loader_cls=TextLoader)
docs = loader.load()
print("로드 된 파일 개수 :", len(docs))

디렉토리 경로'../../Data_Files' 아래 'txt'형식의 파일들을 로드합니다.

로드 된 파일 개수 : 1


In [5]:
for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.metadata['source']}")

1. ../../Data_Files/Crawling_DataFile_MergePage_json_20250512.txt


In [6]:
print(f"청크사이즈{input_chunk_size}, 청크오버랩 사이즈{input_chunk_overlap} 인 text_splitter를 실행합니다.")

청크사이즈1000, 청크오버랩 사이즈50 인 text_splitter를 실행합니다.


In [7]:
texts = text_splitter.split_documents(docs)

In [8]:
print("텍스트 스플릿 된 총 청크 개수: ",len(texts))


텍스트 스플릿 된 총 청크 개수:  630


In [9]:
# 임베딩
from langchain_community.embeddings import HuggingFaceEmbeddings

model_name = 'nlpai-lab/KURE-v1'
# model_name = 'jhgan/ko-sbert-nli'
# model_kwargs = {'device': 'cpu'} # cpu 사용하려면
model_kwargs = {'device': 'mps'} # Mac M1/M2 GPU 사용
encode_kwargs = {'normalize_embeddings': True}
print(f"hf_embedding_model설정, 모델 이름 : {model_name}\n")
hf_embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

hf_embedding_model설정, 모델 이름 : nlpai-lab/KURE-v1



/var/folders/1f/wnkqt1vs7xzfc9mpnzwynpbh0000gn/T/ipykernel_10931/2521201531.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embedding_model = HuggingFaceEmbeddings(


In [13]:
# 임베딩 벡터 경로 지정
# 파일 형식(txt/json)_청크-오버랩_임베딩모델
embedding_directory = './'+filetype+'_'+str(input_chunk_size)+'-'+str(input_chunk_overlap)+'_'+model_name
print(f"임베딩 벡터db 경로: {embedding_directory}")

임베딩 벡터db 경로: ./txt_1000-50_nlpai-lab/KURE-v1


In [14]:
from langchain_chroma import Chroma
# save_directory 가 존재한다면 해당 경로를 db로 설정
if os.path.exists(embedding_directory):
    # 기존 DB 로드
    db = Chroma(persist_directory=embedding_directory, embedding_function=hf_embedding_model)
    print("기존 Chroma DB를 로드했습니다.")
else:
    # 새 DB 생성 및 저장
    db = Chroma.from_documents(texts, hf_embedding_model, persist_directory=embedding_directory)
    print("새로운 Chroma DB를 생성하고 저장했습니다.")

기존 Chroma DB를 로드했습니다.


# 검색, 프롬프트, LLM 설정
- k_num : 검색 개수
- bm_k_num : bm_retriever 의 검색 개수
- retriever1_rate : retriever의 비율(1이하)
- prompt_content : 프롬프트 내용
- LLM_model : LLM모델 (현재 gpt-4o-mini만 사용)


In [15]:
# 문서 검색기 1
k_num = 3

retriever = db.as_retriever(
    search_kwargs = {'k': k_num}
)

# 문서 검색기 2
from langchain_community.retrievers import BM25Retriever
bm_k_num = 3
bm_retriever = BM25Retriever.from_documents(
    documents = docs,
    preprocess_func = okt_tokenize,
)

bm_retriever.k = bm_k_num

# 앙상블 retriever
from langchain.retrievers import EnsembleRetriever
retriever1_rate = 0.5
ensemble_retriever = EnsembleRetriever(
    retrievers = [retriever, bm_retriever],
    weights = [retriever1_rate, 1-retriever1_rate]
)

# 검색 문서 포맷
def format_docs(docs):
    return "\n\n".join(document.page_content for document in docs)

In [16]:
# 프롬프트
from langchain_core.prompts import ChatPromptTemplate

prompt_content = """
            You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If you don't know the answer, just say that you don't know.

            Answer in Korean.

            #Context:
            {context}
            """

few_shot_context = """
JSON 형식의 문서를 전달해줄게. 아래 예시를 참고해서 대답해줘. 대답은 한국어로 해줘.

JSON의 형식은 아래와 같아:

"{{title}}": "제목",
"{{company}}": "회사명",
"{{techStack}}": "요구 기술 스택",
"{{jobCategory}}": "직무 카테고리",
"{{jobLocation}}": "근무지",
"{{minCareer}}": "최소 경력",
"{{maxCareer}}": "최대 경력",
"{{mainWork}}": "주요 업무",
"{{requirements}}": "자격 요건",
"{{preferential}}": "우대 사항",
"{{welfare}}": "복리후생",
"{{education}}": "요구 학력",
"{{closedAt}}": "마감일",
"{{date}}": "등록일"
"{{pride}}" : "특이사항"

예시1:
제목: [챌린저스 QA Manager]
주요 업무: 테스트를 통해 플랜 테스트 문서 작성, 기획 및 분석 단계에서 엣지 케이스를 함께 발굴해요
자격 요건: QA 엔지니어로서 3년 이상의 경력

#Context:
{context}
"""



prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            few_shot_context,
        ), ("human", "{question}"),
    ]
)

In [17]:
from langchain_openai import ChatOpenAI
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

LLM_model="gpt-4o-mini"
llm = ChatOpenAI(
    model = LLM_model,
    temperature = 0,
    streaming = True,
    callbacks = [StreamingStdOutCallbackHandler()],
)

# LLM 체인 형성

In [18]:
# Mac 환경에서만
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

chain = {
    "context" : retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough()
} | prompt | llm | StrOutputParser()

In [20]:
question = 'QA엔지니어를 뽑는 채용공고를 알려줘.'
response = chain.invoke(question)

{
  "title": "[QA 엔지니어 모집]",
  "company": "테크놀로지 주식회사",
  "techStack": "Java, Selenium, JIRA",
  "jobCategory": "QA/테스트",
  "jobLocation": "서울 강남구",
  "minCareer": "3년",
  "maxCareer": "5년",
  "mainWork": "소프트웨어 테스트 계획 수립 및 실행, 결함 관리 및 보고",
  "requirements": "QA 엔지니어로서 3년 이상의 경력, 자동화 테스트 경험",
  "preferential": "Agile 환경에서의 경험, CI/CD 도구 사용 경험",
  "welfare": "4대 보험, 연차, 자기계발비 지원",
  "education": "대졸 이상",
  "closedAt": "2023-12-31",
  "date": "2023-10-01",
  "pride": "유연한 근무 시간과 재택 근무 가능"
}